# Hunting MicroMonsters Bioinformatic Analysis Workflow

## Convert long *.fast5 names to short names

This document shows a method to easely convert long file names from ONT sequencing raw data to a short-consecutive-number set of files.
Each file name will be compound by the container folder name + the number of the file starting from 0.

#### Step-by-step breakdown:

1. find . -type f: Finds all files (not directories) starting from the current directory (.) and going into all subdirectories recursively.

2. while read -r filepath: Processes each file path found by the find command one at a time.

3. dir=$(dirname " $filepath"): Extracts the directory path from the full file path. For example:
    - If filepath is ./subfolder/myfile.txt, dir becomes ./subfolder
    - If filepath is ./myfile.txt, dir becomes .

4. file=$(basename "$filepath"): Extracts just the filename from the full path.
    - From ./subfolder/myfile.txt, file becomes myfile.txt
5. Folder name logic:
    - If the file is in the root directory (dir = "."), use the current working directory's name
    - Otherwise, use the subdirectory's name as the prefix
6. Counters calculation
    - Counts how many files already exist in that directory with the naming pattern foldername_*
    - This ensures proper numbering even if the script is run multiple times

7. Extension handling: Same as your original code - extracts the file extension.

8. File renaming: Moves the file to its new name with the pattern foldername_counter.extension


### Changing the name of each file in one folder with the format "folder_name + number + extension"

Change the name while keeping the original file extensions. This process uses bash kernel.

In [ ]:
#locate the folder with the files
cd ~/path/to/folder
counter=0
folder_name=$(basename "$PWD")
for file in *; do
    if [ -f "$file" ]; then
        # Get file extension
        ext="${file##*.}"
        # Only add extension if file had one
        if [ "$ext" != "$file" ]; then
            mv "$file" "${folder_name}_${counter}.${ext}"
        else
            mv "$file" "${folder_name}_${counter}"
        fi
        ((counter++))
    fi
done

### Changing the name of each file in all subdirectories of a folder with the format "folder_name + number + extension"

A modification from the previous script that can change the name of several subdirectories contained in a main folder.

In [ ]:
# locate the folder with the files
cd ~/path/to/folder

find . -type f | while read -r filepath; do
    # Get the directory and filename
    dir=$(dirname "$filepath")
    file=$(basename "$filepath")
    
    # Get folder name (use parent folder name or current folder for root)
    if [ "$dir" = "." ]; then
        folder_name=$(basename "$PWD")
    else
        folder_name=$(basename "$dir")
    fi
    
    # Count existing renamed files in this directory to continue numbering
    counter=$(find "$dir" -maxdepth 1 -name "${folder_name}_*" | wc -l)
    
    # Get file extension
    ext="${file##*.}"
    
    # Rename the file
    if [ "$ext" != "$file" ]; then
        mv "$filepath" "${dir}/${folder_name}_${counter}.${ext}"
    else
        mv "$filepath" "${dir}/${folder_name}_${counter}"
    fi
done

## Converting the fast5 files to pod5

In this section it is described the process to convert the fast5 files from the sequencing to pod5 to optimized computing performance. The data will be processed downstream using Guppy.

The general workflow for this section is:
1. Environment creation
2. Convert single fasta5 reads into multi-read fast5 reads; using the "ont-fast5-api" module tool.
3. Convert the multiread-fast5 to pod5
4. Basecalling using guppy and the pod5 files

### Fast5-to-pod5 environment creation

Instructions to create the environments and modules needed for each task of the workflow (all commands run on bash kernel).

In [ ]:
# Create the environment
conda create -n fast5-to-pod5 -y

In [ ]:
# Activate environment
conda activate fast5-to-pod5

In [ ]:
# Install ont-fast5-api tool
pip install ont-fast5-api

In [ ]:
# Install pod5 tool
pip install pod5

### Activate the fast5-to-pod5 environment which contains the pod5 module

In [ ]:
# Activate the fast5-to-pod5 conda environment if is not active yet
conda activate fast5-to-pod5

Verify that the environment is active and that the module pod5 is running by looking for the "ont-fast5-api" module

In [ ]:
conda list ont-fast5-api

### Convert single fasta5 reads into multi-read fast5

To create the multi-read files, the tool will create a single file with all the fast5 files in the working folder (4000). The tool requires the input folder path and the output folder. "--recursive" command will look in subdirectories to convert.

In [ ]:
single_to_multi_fast5 \
--input_path ~/path/to/folder/input \
--save_path ~/path/to/folder/ouptut \
--recursive

### Converting multiread-fast5 files into pod5

The newly converted multiread-fast5 will be converted to pod5 files with the pod5 tool. The script is run on bash and includes the command "--one-to-one" to keep the order of the files. The command "--threads" specify the use of all the cores available to speed up the process.

In [ ]:
pod5 convert fast5 \
~/path/to/folder/input/*.fast5 \
--output ~/path/to/folder/output \
--one-to-one ~/path/to/folder/input \
--threads 8

In [ ]:
# Deactivate the fast5-to-pod5 environment once the process is finished
conda deactivate

## Basecalling with Guppy from pod5 files

Now that we have the pod5 files on a folder, we will use Guppy tool for the basecalling. The script is run on bash using the "guppy" environment

Download the guppy file "ont_guppy_cpu_6.5.7-1~focal_amd64.deb" from the ONT community official webpage:
https://nanoporetech.com/software/other/guppy 

In [ ]:
# Create the guppy environment
conda create -n guppy

In [ ]:
# Activate the guppy environment
conda activate guppy

In [ ]:
# Install the *.deb file with guppy basecaller
sudo apt install ./ont_guppy_cpu_6.5.7-1~focal_amd64.deb

In [ ]:
# Verify that the environment is install and working by showing the "version"
guppy_basecaller --version

In [ ]:
# Check the help menu for details
guppy_basecaller -h

In [ ]:
# Execute the guppy module

guppy_basecaller \
  -i ~/path/to/folder/input/pod5_files \
  -s ~/path/to/folder/guppy_output \
  -c dna_r9.4.1_450bps_hac.cfg \
  --num_callers 4 \
  --cpu_threads_per_caller 2

Explanation of the script:

1. guppy_basecaller = calls the guppy module
2. -i = path to input folder
3. -s = path to save folder with fastq data created
4. -c = configuration of the cell flow. In this case, is an old chemistry. The "hac" indicates "high accuracy" (good quality, slow process)
5. --num_callers = number of parallel processes
6. --cpu_threads_per_caller number of cores per caller

The process using only CPU will take some days to finish, depending on the file size.

In [ ]:
# Deactivate the guppy environment
conda deactivate

## Compress the multi-fastq files created into a gz file using gzip for downstream analysis

This is to use the porechop tool

In [ ]:
# compress files using gzip
gzip ~/path/to/file/*.fastq

## Quality analysis

This section is to verify the quality of the fastq files generated, data analysis of the summary_sequencing.txt file generated by guppy.

In [ ]:
# Install nanoplot tool
pip install NanoPlot

In [ ]:
# Create plots from sequencing using the summary file from guppy basecaller
NanoPlot \
--summary sequencing_summary.txt \
--loglength \
-o ~/path/to/folder/output 

## Trimming and QC filter of fastq files

### Trimming with porechop
The trimming of the ONT adapters will be made with the module porechop

In [ ]:
# Create the environment
conda create -n ont-trimming -y

In [ ]:
# Activate environment
conda activate ont-trimming

In [ ]:
# install porechop
mamba install porechop

Trimming the sequences to eliminate the adapters added during the ONT sequencing.

In [ ]:
# Install Filtlong to select the high-quality readings after trimming
conda instal -c bioconda filtlong

In [ ]:
# Run porechop in a for loop to keep multi-fastq trimmed files
for file in *.fastq.gz; do porechop -i "$file" -o ~/path/to/output/file/"trimmed_${file}"; done

### QC filtering
For this section, look into the plots generated by NanoPlot and select the parameters to filter high-quality sequences using Filtlong.
Based on the information, filter the data based on the following:
1. Minimum Quality = 10
2. Minimum length (bp) = 900
3. Discard the worst 10%

In [ ]:
# Install filtlong tool
conda install -c bioconda filtlong -y

#### Filtering with filtlong script for linux
Memory Management
1. 4 parallel jobs maximum: With Filtlong using approximately 1-6GB RAM per process, running 4 jobs simultaneously uses ~24GB, leaving 8GB for the system
2. Memory limits per process: Uses ulimit to cap memory usage per job at 6GB
3. Temporary file management: Uses /tmp for intermediate files and cleans up automatically

CPU Utilization
1. 50% core utilization: Uses 4 out of 8 cores, preventing system overload
2. Staggered job starts: 2-second delay between job launches to prevent resource spikes
3. Active monitoring: Tracks memory usage and job completion in real-time
   
Performance Features
1. Batch processing: Maintains consistent parallel load without overwhelming the system
2. Resource monitoring: Real-time memory usage tracking
3. Error handling: Comprehensive logging and failure detection
4. Automatic cleanup: Removes temporary files and handles interrupted processing

#### Adjust Filtlong parameters:
The following parameters along with the input and output directory should be added to the script:\
KEEP_PERCENT = 90: Keep 90% of bases (by quality)\
TARGET_BASES = 500000000: Target 500M bases output\
MIN_LENGTH = 900: Minimum read length 0.9 kb\
MIN_MEAN_Q = 9: Minimum mean quality score

In [ ]:
#!/bin/bash

# =============================================================================
# FILTLONG PARALLEL PROCESSING SCRIPT - OPTIMIZED FOR 8 CORES, 32GB RAM
# =============================================================================

# Configuration
INPUT_DIR="/path/to/your/fastq/files"
OUTPUT_DIR="/path/to/output/directory"
TEMP_DIR="/tmp/filtlong_processing"

# System-optimized settings for 8 cores, 32GB RAM
MAX_PARALLEL_JOBS=4  # Conservative: 4 parallel jobs to leave headroom
MEMORY_PER_JOB="6G"  # ~6GB per job (4 jobs × 6GB = 24GB, leaving 8GB for system)

# Filtlong parameters (adjust based on your needs)
KEEP_PERCENT=90
TARGET_BASES=500000000  # 500M bases
MIN_LENGTH=1000
MIN_MEAN_Q=7

# =============================================================================
# SETUP AND VALIDATION
# =============================================================================

# Create directories
mkdir -p "$OUTPUT_DIR" "$TEMP_DIR"

# Check available memory
AVAILABLE_RAM=$(free -g | awk '/^Mem:/{print $7}')
if [ "$AVAILABLE_RAM" -lt 20 ]; then
    echo "Warning: Less than 20GB available RAM detected. Consider reducing MAX_PARALLEL_JOBS."
    echo "Available RAM: ${AVAILABLE_RAM}GB"
    read -p "Continue anyway? (y/n): " -n 1 -r
    echo
    if [[ ! $REPLY =~ ^[Yy]$ ]]; then
        exit 1
    fi
fi

echo "=== Filtlong Parallel Processing Started ==="
echo "Input directory: $INPUT_DIR"
echo "Output directory: $OUTPUT_DIR"
echo "Max parallel jobs: $MAX_PARALLEL_JOBS"
echo "Memory per job: $MEMORY_PER_JOB"
echo "=========================================="

# =============================================================================
# PROCESSING FUNCTION
# =============================================================================

process_file() {
    local input_file="$1"
    local job_id="$2"
    
    # Extract filename without path and extension
    local filename=$(basename "$input_file" .fastq.gz)
    local output_file="$OUTPUT_DIR/${filename}_filtered.fastq.gz"
    local temp_output="$TEMP_DIR/${filename}_temp_${job_id}.fastq"
    local log_file="$OUTPUT_DIR/${filename}_filtlong.log"
    
    echo "[Job $job_id] Starting processing: $filename"
    echo "[Job $job_id] $(date): Processing started" > "$log_file"
    
    # Monitor memory usage for this process
    local start_time=$(date +%s)
    
    # Run Filtlong with resource limits
    {
        # Use ulimit to restrict memory usage per process (convert GB to KB)
        local memory_limit_kb=$((${MEMORY_PER_JOB%G} * 1024 * 1024))
        ulimit -v $memory_limit_kb
        
        # Run filtlong and capture both stdout and stderr
        filtlong \
            --keep_percent "$KEEP_PERCENT" \
            --target_bases "$TARGET_BASES" \
            --min_length "$MIN_LENGTH" \
            --min_mean_q "$MIN_MEAN_Q" \
            --verbose \
            "$input_file" > "$temp_output" 2>>"$log_file"
        
        # Check if filtlong succeeded
        if [ $? -eq 0 ] && [ -s "$temp_output" ]; then
            # Compress the output
            gzip -c "$temp_output" > "$output_file"
            rm "$temp_output"  # Clean up temporary file
            
            local end_time=$(date +%s)
            local duration=$((end_time - start_time))
            
            echo "[Job $job_id] $(date): Processing completed in ${duration}s" >> "$log_file"
            echo "[Job $job_id] Completed: $filename (${duration}s)"
        else
            echo "[Job $job_id] ERROR: Failed to process $filename" | tee -a "$log_file"
            rm -f "$temp_output" "$output_file"  # Clean up on failure
            return 1
        fi
    }
}

# Export function and variables for background processes
export -f process_file
export OUTPUT_DIR TEMP_DIR KEEP_PERCENT TARGET_BASES MIN_LENGTH MIN_MEAN_Q

# =============================================================================
# PARALLEL PROCESSING WITH RESOURCE MANAGEMENT
# =============================================================================

# Find all FASTQ.gz files
mapfile -t fastq_files < <(find "$INPUT_DIR" -name "*.fastq.gz" -type f)

if [ ${#fastq_files[@]} -eq 0 ]; then
    echo "No FASTQ.gz files found in $INPUT_DIR"
    exit 1
fi

echo "Found ${#fastq_files[@]} FASTQ.gz files to process"

# Process files in batches to manage memory
job_counter=0
active_jobs=()
failed_files=()

for file in "${fastq_files[@]}"; do
    # Wait if we've reached the maximum number of parallel jobs
    while [ ${#active_jobs[@]} -ge $MAX_PARALLEL_JOBS ]; do
        # Check which jobs have completed
        new_active_jobs=()
        for pid in "${active_jobs[@]}"; do
            if kill -0 "$pid" 2>/dev/null; then
                new_active_jobs+=("$pid")
            else
                # Job completed, check exit status
                wait "$pid"
                if [ $? -ne 0 ]; then
                    echo "Warning: A processing job failed"
                fi
            fi
        done
        active_jobs=("${new_active_jobs[@]}")
        
        # Small delay to prevent excessive CPU usage in the wait loop
        sleep 1
        
        # Memory monitoring
        used_memory=$(free | awk '/^Mem:/{printf "%.1f", $3/$2 * 100.0}')
        echo "Memory usage: ${used_memory}% | Active jobs: ${#active_jobs[@]}/$MAX_PARALLEL_JOBS"
    done
    
    # Start new job
    ((job_counter++))
    echo "Starting job $job_counter for: $(basename "$file")"
    
    process_file "$file" "$job_counter" &
    job_pid=$!
    active_jobs+=("$job_pid")
    
    # Brief pause between job starts to prevent resource spikes
    sleep 2
done

# Wait for all remaining jobs to complete
echo "Waiting for remaining jobs to complete..."
for pid in "${active_jobs[@]}"; do
    wait "$pid"
    if [ $? -ne 0 ]; then
        echo "Warning: Job with PID $pid failed"
    fi
done

# =============================================================================
# CLEANUP AND SUMMARY
# =============================================================================

echo "=== Processing Summary ==="

# Count successful outputs
successful_files=$(find "$OUTPUT_DIR" -name "*_filtered.fastq.gz" -type f | wc -l)
echo "Successfully processed: $successful_files/${#fastq_files[@]} files"

# Check for failed processing
if [ -d "$TEMP_DIR" ]; then
    remaining_temp_files=$(find "$TEMP_DIR" -name "*temp*.fastq" -type f 2>/dev/null | wc -l)
    if [ "$remaining_temp_files" -gt 0 ]; then
        echo "Warning: $remaining_temp_files temporary files remain (possible failures)"
    fi
    
    # Clean up temporary directory
    rm -rf "$TEMP_DIR"
fi

# Display log summary
echo "Log files created in: $OUTPUT_DIR/*_filtlong.log"

# Final memory check
final_memory=$(free -h | awk '/^Mem:/{print "Used: " $3 ", Available: " $7}')
echo "Final memory status: $final_memory"

echo "=== Processing Complete ==="


In [ ]:
# To run the script in bash
chmod +x filtlong_parallel.sh
./filtlong_parallel.sh

#### Enhanced script with QC filtering combined with Post-filtering QC analysis with NanoPlot

This script adds a post-filtering QC analysis to the files generated by filtlong using nanoplot.
The output structure of the file directory will be as follows:

In [ ]:
output_directory/
├── filtered/                           # Individual filtered files
│   ├── sample1_filtered.fastq.gz
│   ├── sample2_filtered.fastq.gz
│   └── sample3_filtered.fastq.gz
├── combined_qc/                        # Single combined QC analysis
│   ├── Combined_Dataset_NanoPlot-report.html  # Main interactive report
│   ├── NanoStats.txt                          # Combined statistics
│   ├── Combined_Dataset_combined_summary.txt  # Enhanced summary
│   └── individual_plots/                      # PNG and HTML plots
├── logs/                               # Processing logs
│   ├── sample1_filtlong.log
│   ├── sample2_filtlong.log
│   └── combined_nanoplot.log           # Combined QC log
└── pipeline_summary.txt                # Overall pipeline summary


In [ ]:
#!/bin/bash

# =============================================================================
# ENHANCED FILTLONG + NANOPLOT COMBINED MODE PIPELINE - OPTIMIZED FOR 8 CORES, 32GB RAM
# =============================================================================

# Configuration
INPUT_DIR="/path/to/your/fastq/files"
OUTPUT_DIR="/path/to/output/directory"
TEMP_DIR="/tmp/filtlong_nanoplot_processing"

# System-optimized settings for 8 cores, 32GB RAM
MAX_PARALLEL_JOBS=4  # Back to 4 since we're only running NanoPlot once at the end
MEMORY_PER_JOB="6G"  # Memory per Filtlong job

# Filtlong parameters
KEEP_PERCENT=90
TARGET_BASES=500000000
MIN_LENGTH=1000
MIN_MEAN_Q=7

# NanoPlot Combined Mode parameters
NANOPLOT_THREADS=4      # Use more threads since it's a single NanoPlot job
NANOPLOT_PLOTS="kde hex dot"  # Plot types to generate
NANOPLOT_PREFIX="Combined_Dataset"  # Prefix for combined analysis
ENABLE_QC=true          # Set to false to skip QC step

# =============================================================================
# SETUP AND VALIDATION
# =============================================================================

# Create directories
mkdir -p "$OUTPUT_DIR"/{filtered,combined_qc,logs} "$TEMP_DIR"

# Check system resources
AVAILABLE_RAM=$(free -g | awk '/^Mem:/{print $7}')
AVAILABLE_CORES=$(nproc)

echo "=== Enhanced Filtlong + NanoPlot Combined Mode Pipeline Started ==="
echo "Available RAM: ${AVAILABLE_RAM}GB"
echo "Available CPU cores: $AVAILABLE_CORES"
echo "Max parallel Filtlong jobs: $MAX_PARALLEL_JOBS"
echo "NanoPlot Combined Mode: $ENABLE_QC"
echo "=========================================="

# Validate system requirements
if [ "$AVAILABLE_RAM" -lt 20 ]; then
    echo "Warning: Less than 20GB available RAM. Consider reducing MAX_PARALLEL_JOBS."
    read -p "Continue anyway? (y/n): " -n 1 -r
    echo
    if [[ ! $REPLY =~ ^[Yy]$ ]]; then
        exit 1
    fi
fi

# Check if NanoPlot is available
if ! command -v NanoPlot &> /dev/null && [ "$ENABLE_QC" = true ]; then
    echo "Error: NanoPlot not found. Install it with: pip install NanoPlot"
    echo "Or set ENABLE_QC=false to skip QC analysis"
    exit 1
fi

# =============================================================================
# PROCESSING FUNCTIONS
# =============================================================================

# Function to process a single file through Filtlong
process_filtlong() {
    local input_file="$1"
    local job_id="$2"
    
    local filename=$(basename "$input_file" .fastq.gz)
    local output_file="$OUTPUT_DIR/filtered/${filename}_filtered.fastq.gz"
    local temp_output="$TEMP_DIR/${filename}_temp_${job_id}.fastq"
    local log_file="$OUTPUT_DIR/logs/${filename}_filtlong.log"
    
    echo "[Filtlong Job $job_id] Starting: $filename"
    echo "[Job $job_id] $(date): Filtlong processing started" > "$log_file"
    
    local start_time=$(date +%s)
    
    {
        # Memory limit
        local memory_limit_kb=$((${MEMORY_PER_JOB%G} * 1024 * 1024))
        ulimit -v $memory_limit_kb
        
        # Run Filtlong
        filtlong \
            --keep_percent "$KEEP_PERCENT" \
            --target_bases "$TARGET_BASES" \
            --min_length "$MIN_LENGTH" \
            --min_mean_q "$MIN_MEAN_Q" \
            --verbose \
            "$input_file" > "$temp_output" 2>>"$log_file"
        
        if [ $? -eq 0 ] && [ -s "$temp_output" ]; then
            gzip -c "$temp_output" > "$output_file"
            rm "$temp_output"
            
            local end_time=$(date +%s)
            local duration=$((end_time - start_time))
            
            echo "[Job $job_id] $(date): Filtlong completed in ${duration}s" >> "$log_file"
            echo "[Filtlong Job $job_id] Completed: $filename (${duration}s)"
            return 0
        else
            echo "[Filtlong Job $job_id] ERROR: Failed to process $filename" | tee -a "$log_file"
            rm -f "$temp_output" "$output_file"
            return 1
        fi
    }
}

# Function to run NanoPlot in Combined Mode on all filtered files
run_combined_nanoplot() {
    echo "=== Starting NanoPlot Combined Mode Analysis ==="
    
    # Find all filtered files
    local filtered_files=("$OUTPUT_DIR/filtered"/*_filtered.fastq.gz)
    
    if [ ${#filtered_files[@]} -eq 0 ]; then
        echo "No filtered files found for combined QC analysis"
        return 1
    fi
    
    echo "Found ${#filtered_files[@]} filtered files for combined analysis"
    
    local qc_output_dir="$OUTPUT_DIR/combined_qc"
    local qc_log_file="$OUTPUT_DIR/logs/combined_nanoplot.log"
    
    echo "$(date): NanoPlot Combined Mode started" > "$qc_log_file"
    echo "Input files:" >> "$qc_log_file"
    for file in "${filtered_files[@]}"; do
        echo "  - $(basename "$file")" >> "$qc_log_file"
    done
    
    local start_time=$(date +%s)
    
    {
        # Memory limit for NanoPlot (use more memory since it's processing all files)
        local memory_limit_kb=$((24 * 1024 * 1024))  # 24GB for combined processing
        ulimit -v $memory_limit_kb
        
        # Run NanoPlot in Combined Mode[159][166][208]
        NanoPlot \
            --fastq "${filtered_files[@]}" \
            --outdir "$qc_output_dir" \
            --prefix "${NANOPLOT_PREFIX}_" \
            --threads "$NANOPLOT_THREADS" \
            --plots $NANOPLOT_PLOTS \
            --N50 \
            --loglength \
            --drop_outliers \
            --verbose 2>>"$qc_log_file"
        
        if [ $? -eq 0 ]; then
            # Generate enhanced summary statistics
            local stats_file="$qc_output_dir/${NANOPLOT_PREFIX}_combined_summary.txt"
            {
                echo "=== Combined QC Summary for All Filtered Files ==="
                echo "Generated on: $(date)"
                echo "Analysis mode: Combined Mode (all files processed as single dataset)"
                echo "Number of input files: ${#filtered_files[@]}"
                echo "Output directory: $qc_output_dir"
                echo ""
                
                echo "=== Input Files Processed ==="
                for file in "${filtered_files[@]}"; do
                    local file_size=$(du -h "$file" | cut -f1)
                    echo "  - $(basename "$file") (${file_size})"
                done
                echo ""
                
                # Extract key stats from NanoStats.txt if available[160]
                if [ -f "$qc_output_dir/NanoStats.txt" ]; then
                    echo "=== Combined Dataset Statistics ==="
                    cat "$qc_output_dir/NanoStats.txt"
                    echo ""
                fi
                
                echo "=== Available Reports and Plots ==="
                echo "Main HTML report: ${NANOPLOT_PREFIX}_NanoPlot-report.html"
                echo "Statistics file: NanoStats.txt"
                echo "Individual plots: Available as PNG and HTML files"
                
            } > "$stats_file"
            
            local end_time=$(date +%s)
            local duration=$((end_time - start_time))
            
            echo "$(date): NanoPlot Combined Mode completed in ${duration}s" >> "$qc_log_file"
            echo "[Combined QC] Completed: All filtered files analyzed in ${duration}s"
            echo "[Combined QC] Main report: $qc_output_dir/${NANOPLOT_PREFIX}_NanoPlot-report.html"
            return 0
        else
            echo "[Combined QC] ERROR: NanoPlot Combined Mode failed" | tee -a "$qc_log_file"
            return 1
        fi
    }
}

# Export functions
export -f process_filtlong
export OUTPUT_DIR TEMP_DIR KEEP_PERCENT TARGET_BASES MIN_LENGTH MIN_MEAN_Q MEMORY_PER_JOB

# =============================================================================
# PARALLEL FILTLONG PROCESSING
# =============================================================================

echo "=== Phase 1: Parallel Filtlong Processing ==="

# Find all FASTQ.gz files
mapfile -t fastq_files < <(find "$INPUT_DIR" -name "*.fastq.gz" -type f)

if [ ${#fastq_files[@]} -eq 0 ]; then
    echo "No FASTQ.gz files found in $INPUT_DIR"
    exit 1
fi

echo "Found ${#fastq_files[@]} FASTQ.gz files to process with Filtlong"

# Process files with controlled parallelism
job_counter=0
active_jobs=()
failed_files=()

for file in "${fastq_files[@]}"; do
    # Wait if we've reached the maximum number of parallel jobs
    while [ ${#active_jobs[@]} -ge $MAX_PARALLEL_JOBS ]; do
        new_active_jobs=()
        for pid in "${active_jobs[@]}"; do
            if kill -0 "$pid" 2>/dev/null; then
                new_active_jobs+=("$pid")
            else
                wait "$pid"
                if [ $? -ne 0 ]; then
                    echo "Warning: Filtlong job with PID $pid failed"
                fi
            fi
        done
        active_jobs=("${new_active_jobs[@]}")
        
        # Resource monitoring
        used_memory=$(free | awk '/^Mem:/{printf "%.1f", $3/$2 * 100.0}')
        echo "Filtlong Status: Memory ${used_memory}% | Active jobs: ${#active_jobs[@]}/$MAX_PARALLEL_JOBS"
        
        sleep 3
    done
    
    # Start new Filtlong job
    ((job_counter++))
    echo "Starting Filtlong job $job_counter for: $(basename "$file")"
    
    process_filtlong "$file" "$job_counter" &
    job_pid=$!
    active_jobs+=("$job_pid")
    
    # Staggered start
    sleep 2
done

# Wait for all Filtlong jobs to complete
echo "Waiting for remaining Filtlong jobs to complete..."
for pid in "${active_jobs[@]}"; do
    wait "$pid"
    if [ $? -ne 0 ]; then
        echo "Warning: Filtlong job with PID $pid failed"
    fi
done

echo "=== Phase 1 Complete: Filtlong Processing Finished ==="

# =============================================================================
# COMBINED NANOPLOT QC ANALYSIS
# =============================================================================

if [ "$ENABLE_QC" = true ]; then
    echo ""
    echo "=== Phase 2: Combined NanoPlot QC Analysis ==="
    
    # Count successful Filtlong outputs
    successful_filtered=$(find "$OUTPUT_DIR/filtered" -name "*_filtered.fastq.gz" -type f | wc -l)
    
    if [ "$successful_filtered" -eq 0 ]; then
        echo "No successfully filtered files found. Skipping QC analysis."
    else
        echo "Processing $successful_filtered filtered files in Combined Mode..."
        
        # Run combined NanoPlot analysis
        if run_combined_nanoplot; then
            echo "Combined QC analysis completed successfully!"
        else
            echo "Combined QC analysis failed. Check logs for details."
        fi
    fi
else
    echo "=== Phase 2 Skipped: QC analysis disabled ==="
fi

# =============================================================================
# FINAL SUMMARY AND CLEANUP
# =============================================================================

echo ""
echo "=== Processing Complete - Generating Final Summary ==="

# Count results
successful_filtered=$(find "$OUTPUT_DIR/filtered" -name "*_filtered.fastq.gz" -type f | wc -l)
qc_report_exists=false

if [ "$ENABLE_QC" = true ] && [ -f "$OUTPUT_DIR/combined_qc/${NANOPLOT_PREFIX}_NanoPlot-report.html" ]; then
    qc_report_exists=true
fi

# Generate comprehensive summary report
summary_report="$OUTPUT_DIR/pipeline_summary.txt"
{
    echo "=== Enhanced Filtlong + NanoPlot Combined Mode Pipeline Summary ==="
    echo "Execution date: $(date)"
    echo "Input directory: $INPUT_DIR"
    echo "Output directory: $OUTPUT_DIR"
    echo ""
    echo "=== Processing Parameters ==="
    echo "Filtlong settings:"
    echo "  - Keep percent: $KEEP_PERCENT%"
    echo "  - Target bases: $TARGET_BASES"
    echo "  - Minimum length: $MIN_LENGTH bp"
    echo "  - Minimum mean quality: $MIN_MEAN_Q"
    echo "  - Max parallel jobs: $MAX_PARALLEL_JOBS"
    echo ""
    echo "NanoPlot Combined Mode settings:"
    echo "  - QC enabled: $ENABLE_QC"
    echo "  - Analysis mode: Combined (all files as single dataset)"
    echo "  - Threads: $NANOPLOT_THREADS"
    echo "  - Plot types: $NANOPLOT_PLOTS"
    echo "  - Prefix: $NANOPLOT_PREFIX"
    echo ""
    echo "=== Results Summary ==="
    echo "Total input files: ${#fastq_files[@]}"
    echo "Successfully filtered: $successful_filtered"
    echo "Combined QC analysis: $($qc_report_exists && echo "✓ Success" || echo "✗ Failed/Skipped")"
    echo ""
    echo "=== Output Structure ==="
    echo "Filtered reads: $OUTPUT_DIR/filtered/"
    if [ "$ENABLE_QC" = true ]; then
        echo "Combined QC report: $OUTPUT_DIR/combined_qc/"
    fi
    echo "Log files: $OUTPUT_DIR/logs/"
    echo ""
    
    if [ "$qc_report_exists" = true ]; then
        echo "=== Combined Dataset QC Statistics ==="
        if [ -f "$OUTPUT_DIR/combined_qc/NanoStats.txt" ]; then
            echo "Key metrics from combined analysis:"
            grep -E "(Mean read length|Median read length|N50|Total bases|Number of reads|Mean read quality)" "$OUTPUT_DIR/combined_qc/NanoStats.txt" 2>/dev/null || echo "Stats extraction failed"
        fi
        echo ""
    fi
    
    echo "=== Individual File Processing Status ==="
    for file in "${fastq_files[@]}"; do
        filename=$(basename "$file" .fastq.gz)
        filtered_file="$OUTPUT_DIR/filtered/${filename}_filtered.fastq.gz"
        
        echo "File: $filename"
        if [ -f "$filtered_file" ]; then
            filtered_size=$(du -h "$filtered_file" | cut -f1)
            echo "  ✓ Filtered: ${filtered_size}"
        else
            echo "  ✗ Filtering failed"
        fi
    done
    
} > "$summary_report"

# Display key information
echo ""
echo "=== Output Locations ==="
echo "📁 Filtered reads: $OUTPUT_DIR/filtered/"
if [ "$qc_report_exists" = true ]; then
    echo "📊 Combined QC report: $OUTPUT_DIR/combined_qc/${NANOPLOT_PREFIX}_NanoPlot-report.html"
    echo "📈 Combined statistics: $OUTPUT_DIR/combined_qc/NanoStats.txt"
    echo "📋 QC summary: $OUTPUT_DIR/combined_qc/${NANOPLOT_PREFIX}_combined_summary.txt"
fi
echo "📋 Log files: $OUTPUT_DIR/logs/"
echo "📄 Pipeline summary: $summary_report"

# Cleanup temporary directory
rm -rf "$TEMP_DIR"

# Final system status
final_memory=$(free -h | awk '/^Mem:/{print "Used: " $3 ", Available: " $7}')
echo ""
echo "Final memory status: $final_memory"
echo "=== Pipeline Complete ==="

if [ "$qc_report_exists" = true ]; then
    echo ""
    echo "🎯 **MAIN RESULT**: Open the combined QC report in your browser:"
    echo "   $OUTPUT_DIR/combined_qc/${NANOPLOT_PREFIX}_NanoPlot-report.html"
    echo ""
    echo "This report contains comprehensive quality analysis of ALL filtered files"
    echo "processed as a single combined dataset with unified statistics and plots."
fi


In [ ]:
# Deactivate environment
conda deactivate